# NumPy en contexto e introducción a Pandas

## Geocomputación (1GEO20) — PUCP, 2026-II — Semana 2 

Este notebook cubre broadcasting y masks de NumPy aplicados a datos reales, y una primera introducción a Pandas.

**Datos:** temperatura superficial promedio mensual, Perú, 1940-2026. Contiene información modificada de Copernicus Climate Change Service, con procesamiento de Our World in Data (ERA5). Fuente: https://ourworldindata.org/grapher/average-monthly-surface-temperature, licenciado bajo CC BY 4.0.

**Cómo usar este notebook:** completa las celdas marcadas como Ejercicio. Ejecuta cada celda con `Shift + Enter`.

---

## 1. Cargar datos reales con Pandas

Hasta ahora trabajamos con arrays y listas creados a mano. Los datos reales normalmente llegan en archivos, y una de las formas más comunes de leerlos es con Pandas.

**Nota importante:** este sitio requiere identificarse con un encabezado (`User-Agent`) para aceptar la solicitud. Sin ese detalle, la descarga puede fallar.

In [ ]:
import pandas as pd
import numpy as np

url = (
    "https://ourworldindata.org/grapher/average-monthly-surface-temperature.csv"
    "?v=1&csvType=filtered&useColumnShortNames=true&tab=chart&country=PER"
)
headers = {"User-Agent": "Curso 1GEO20 - PUCP"}

df = pd.read_csv(url, storage_options=headers)
df.head()

In [ ]:
# Agregar una columna con formato fecha
df["fecha"] = pd.to_datetime(df["Month"], format="%Y-%m")
df["year"] = df["fecha"].dt.year

Antes de seguir, dos preguntas básicas exploratorias: ¿cuántas filas hay?, ¿qué significa cada columna?

In [ ]:
print(df.shape)
print(df.columns.tolist())

### Ejercicio 1
1. ¿Cuántas filas tiene el DataFrame? ¿A cuántos años de datos corresponde, aproximadamente (12 meses por año)?
2. La columna de temperatura probablemente tenga otro nombre. Identifica su nominación real revisando `df.columns`.


In [ ]:
# 1.


In [ ]:
# 2.


## 2. Broadcasting en contexto

Extraer la columna de temperatura como un array de NumPy, y calcular el promedio histórico de todo el período.

In [ ]:
temp = df["temperature_2m"].to_numpy()
promedio_historico = temp.mean()
print("Promedio histórico:", promedio_historico)
print("Cantidad de meses:", temp.shape)

Restar ese único número a todos los valores del array. Esto se llama "broadcasting": un valor de forma `()` (escalar) se "extiende" para operar con un array de forma mucho más grande.

In [ ]:
anomalia = temp - promedio_historico
print(anomalia[:12])   # anomalía de los primeros 12 meses del registro

Ese resultado, `dato - climatología`, es lo que se conoce como **anomalía**: cuánto se aparta cada mes del promedio de largo plazo.

### Ejercicio 2
1. ¿Cuál es el valor máximo y mínimo de `anomalia`? ¿A qué corresponde, un mes más cálido o más frío que el promedio?
2. En vez de restar el promedio de todo el período, calcula el promedio de los últimos 12 meses del array y réstalo a esos mismos 12 valores. ¿La anomalía resultante es distinta a la que obtendrías usando el promedio histórico completo?

In [ ]:
# 1.


In [ ]:
# 2.


## 3. Masks en contexto



Cuando se compara un array completo con un número, se obtiene un array del mismo tamaño, con `True` o `False` en cada posición.

In [ ]:
ejemplo = np.array([10, 25, 30, 15])
print(ejemplo > 20)


A ese array de `True`/`False` se le llama **máscara** (mask). Usarlo para indexar el array
original devuelve solo los elementos donde la máscara es `True`.

In [ ]:
print(ejemplo[ejemplo > 20])


### Leer una línea con dos operadores: de derecha a izquierda

Una línea puede combinar una comparación (`>`, `<`, `<=`, `==`...) con una asignación (`=`).
Python siempre resuelve primero el lado derecho del `=`, y recién después guarda
ese resultado en la variable de la izquierda.

In [ ]:
es_mayor = 10 > 5   # primero se evalúa 10 > 5 (da True), luego se guarda en es_mayor
print(es_mayor)


Con indexado booleano, un array de condiciones (`True`/`False`) selecciona solo los valores que cumplen una condición.

In [ ]:
umbral = 27
meses_calidos = temp[temp > umbral]
print("Cantidad de meses sobre", umbral, "°C:", meses_calidos.shape[0])
print(meses_calidos[:10])

Si en vez de extraer solo los valores que cumplen la condición se quiere conservar la forma original del array (por ejemplo, para saber en qué posición ocurrió cada uno), `np.ma.MaskedArray` oculta los valores que no cumplen la condición sin eliminarlos del array.

In [ ]:
mask = temp <= umbral   # True donde se debe ocultar
masked_temp = np.ma.MaskedArray(temp, mask)
print(masked_temp[:12])

### Ejercicio 3
1. ¿Cuántos meses del registro completo tuvieron una anomalía positiva (usa el array `anomalia` del Ejercicio 2)?
2. ¿Cuál es el promedio de temperatura únicamente de los meses con anomalía positiva?

In [ ]:
# 1.


In [ ]:
# 2.


## 4. De NumPy a Pandas

Todo lo anterior trabajó sobre un solo array, la columna de temperatura, separada del resto de la información (fechas, país). Un DataFrame de Pandas es como un diccionario de arrays de NumPy (uno por columna, todos alineados por fila). Por eso `df["temperature_2m"]` usa la misma sintaxis que se usó con diccionarios en la Semana 1.

La ventaja de trabajar con el DataFrame completo, en vez de arrays sueltos, es que las columnas se mantienen alineadas: filtrar o calcular sobre una también se aplica a las demás (por ejemplo, la fecha exacta de cada valor de temperatura).

## 5. Explorar y filtrar el DataFrame completo

In [ ]:
df["temperature_2m"].describe()

El mismo filtrado por condición booleana que usaste en NumPy funciona igual sobre un DataFrame completo, y mantiene las demás columnas alineadas:

In [ ]:
meses_calidos_df = df[df["temperature_2m"] > umbral]
meses_calidos_df.head()

### Ejercicio 4
1. Filtra el DataFrame para quedarte solo con los meses de un año específico (utiliza la columna creada ['year'] ).
2. ¿Cuál fue la temperatura promedio de ese año?
3. Compara ese promedio con el promedio histórico completo (`promedio_historico`, calculado en la Sección 2). ¿Fue un año más cálido o más frío que el promedio?

In [ ]:
# 1.


In [ ]:
# 2.


In [ ]:
# 3.


## 6. Más ejercicios aplicados

### Ejercicio 5
Retoma la función `a_celsius_desde_fahrenheit` (o `a_celsius`) del notebook anterior. Aplícala a un valor de la columna `temperature_2m` (por ejemplo, el primero) simulando que ese dato hubiera llegado en Fahrenheit por error. ¿El resultado tiene sentido físico para un clima como el de Perú?

In [ ]:
# Ejercicio 5


### Ejercicio 6
1. Identifica el mes más cálido y el mes más frío de todo el registro (columna de fecha y valor de temperatura de ambos).
2. Usando una máscara booleana, calcula cuántos meses de la última década (desde 2016 en adelante) tuvieron una temperatura por encima del promedio histórico completo (`promedio_historico`).

In [ ]:
# 1.


In [ ]:
# 2.
